In [1]:
# Pipeline Configuration
RUN_MODE = "production"  # "test" or "production"
TEST_SAMPLE_SIZE = 10
MAX_WORKERS = 4
CHUNK_SIZE = 500
ENABLE_CACHE = True
ENABLE_CHECKPOINT = True
PIPELINE_VERSION = "1.0.0"


In [2]:
# !pip -q install geopandas rasterio rioxarray pystac-client planetary-computer odc-stac shapely pyproj xarray folium leafmap



In [3]:
import geopandas as gpd
import pandas as pd
import numpy as np

from shapely.geometry import Point

import rasterio
import rioxarray

import planetary_computer
import pystac_client



In [4]:
import os

folders = [
    "../data",
    "../data/raw",
    "../data/processed",
    "../data/features",
    "../data/metadata",
    "../data/final",
    "../data/lucas",
    "../data/sentinel",
    "../data/weather",
    "../data/soilgrids",
    "../outputs",
    "../outputs/csv",
    "../outputs/maps",
    "../outputs/figures",
    "../outputs/reports",
    "../outputs/metrics",
    "../outputs/learning_curves",
    "../outputs/feature_importance",
    "../outputs/confusion_matrix",
    "../models",
    "../models/machine_learning",
    "../models/deep_learning",
    "../models/ensemble",
    "../models/best",
    "../models/experimental"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"Created: {folder}")




Created: ../data
Created: ../data/raw
Created: ../data/processed
Created: ../data/features
Created: ../data/metadata
Created: ../data/final
Created: ../data/lucas
Created: ../data/sentinel
Created: ../data/weather
Created: ../data/soilgrids
Created: ../outputs
Created: ../outputs/csv
Created: ../outputs/maps
Created: ../outputs/figures
Created: ../outputs/reports
Created: ../outputs/metrics
Created: ../outputs/learning_curves
Created: ../outputs/feature_importance
Created: ../outputs/confusion_matrix
Created: ../models
Created: ../models/machine_learning
Created: ../models/deep_learning
Created: ../models/ensemble
Created: ../models/best
Created: ../models/experimental


In [5]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

print("Connected Successfully")



Connected Successfully


In [6]:
# # collections = catalog.get_collections()

# # for c in collections:
    # # print(c.id)


print("Skipping collections listing for speed.")




Skipping collections listing for speed.


In [7]:
from shapely.geometry import Point

lon = 31.083
lat = 30.563

point = Point(lon, lat)

buffer = point.buffer(0.001)

geometry = buffer.__geo_interface__

geometry



{'type': 'Polygon',
 'coordinates': (((31.084, 30.563),
   (31.08399518472667, 30.562901982859668),
   (31.083980785280403, 30.562804909677983),
   (31.08395694033573, 30.562709715322743),
   (31.08392387953251, 30.562617316567632),
   (31.083881921264346, 30.56252860326317),
   (31.0838314696123, 30.56244442976698),
   (31.08377301045336, 30.562365606715836),
   (31.083707106781183, 30.562292893218814),
   (31.08363439328416, 30.562226989546637),
   (31.083555570233017, 30.562168530387698),
   (31.083471396736826, 30.56211807873565),
   (31.083382683432365, 30.562076120467488),
   (31.083290284677254, 30.562043059664266),
   (31.083195090322015, 30.562019214719594),
   (31.08309801714033, 30.562004815273326),
   (31.083, 30.561999999999998),
   (31.082901982859667, 30.562004815273326),
   (31.082804909677982, 30.562019214719594),
   (31.082709715322743, 30.562043059664266),
   (31.08261731656763, 30.562076120467488),
   (31.08252860326317, 30.56211807873565),
   (31.08244442976698, 30

In [8]:
import requests
import os

def download_soil_layer(layer, depth="0-5cm", resolution="1000m"):
    filename = f"../data/raw/soilgrids/{layer}.tif"
    if os.path.exists(filename):
        print(layer, "already exists ✔ (skipping download)")
        return filename
    url = f"https://files.isric.org/soilgrids/latest/data_aggregated/{resolution}/{layer}/{layer}_{depth}_mean_{resolution}.tif"
    print("Downloading", layer, "from", url)
    r = requests.get(url)
    if r.status_code != 200:
        print("Download failed:", url)
        return None
    with open(filename,"wb") as f:
        f.write(r.content)
    print(layer, "downloaded")
    return filename




In [9]:
import requests

lat = 30.563
lon = 31.083

url = f"https://rest.isric.org/soilgrids/v2.0/properties/query?lon={lon}&lat={lat}"

response = requests.get(url)

print("Status Code:", response.status_code)
print("Content Type:", response.headers.get("content-type"))
print(response.text[:500])



Status Code: 504
Content Type: text/html
<html>
<head><title>504 Gateway Time-out</title></head>
<body>
<center><h1>504 Gateway Time-out</h1></center>
<hr><center>nginx</center>
</body>
</html>



In [10]:
# !pip install rasterio geopandas pyproj



In [11]:
import requests

url = "https://files.isric.org/soilgrids/latest/data/phh2o/"

r = requests.get(url)

print(r.status_code)
print(r.text[:500])



200
<!DOCTYPE html>
<html>
<head>
  <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
  <meta name="generator" content="WsgiDAV/4.3.3">
  <title>ISRIC - Index of /soilgrids/latest/data/phh2o/ </title>
  <link rel="shortcut icon" href="/:dir_browser/favicon.ico">
  <link rel="stylesheet" href="/:dir_browser/style.css" />
  <script defer src="/:dir_browser/script.js"></script>
  <style type="text/css"> A {behavior: url(#default#AnchorClick);} </style>
</head>

<body onload="onLoad()"


In [12]:
import requests

url = "https://files.isric.org/soilgrids/latest/data/phh2o/"

response = requests.get(url)

print(response.status_code)
print(response.text[:1000])



200
<!DOCTYPE html>
<html>
<head>
  <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
  <meta name="generator" content="WsgiDAV/4.3.3">
  <title>ISRIC - Index of /soilgrids/latest/data/phh2o/ </title>
  <link rel="shortcut icon" href="/:dir_browser/favicon.ico">
  <link rel="stylesheet" href="/:dir_browser/style.css" />
  <script defer src="/:dir_browser/script.js"></script>
  <style type="text/css"> A {behavior: url(#default#AnchorClick);} </style>
</head>

<body onload="onLoad()">

  <h1>
    <img class="logo" alt="ISRIC" title="ISRIC" src="/:dir_browser/logo.png">
    Webdav - Index of /soilgrids/latest/data/phh2o/
  </h1>
<p>
This page lists all resources made available through ISRIC - World Soil Information's Webdav functionality. 
This URL can also be accessed via the <a href="https://www.isric.org/explore/soilgrids/soilgrids-access">Webdav protocol</a>.
</p><p>
These resources are described in our data catalogue at <a href="https://data.isric.org">data.isric.or

In [13]:
# !pip install rasterio



In [14]:
# import rasterio

# url = "https://files.isric.org/soilgrids/latest/data/phh2o/phh2o_0-5cm_mean.vrt"

# src = rasterio.open(url)

# print(src)
# print(src.crs)
# print(src.bounds)



In [15]:
# from rasterio.transform import rowcol

# lat = 30.563
# lon = 31.083

# row, col = rowcol(
#     src.transform,
#     lon,
#     lat
# )

# value = src.read(1)[row, col]

# print("Raw pH value:", value)



In [16]:
# import rasterio

# url = "https://files.isric.org/soilgrids/latest/data/phh2o/phh2o_0-5cm_mean.vrt"

# src = rasterio.open(url)

# print("Opened Successfully")
# print(src.crs)
# print(src.bounds)



In [17]:
# from rasterio.transform import rowcol

# lat = 30.563
# lon = 31.083

# row, col = rowcol(
#     src.transform,
#     lon,
#     lat
# )

# value = src.read(1)[row, col]

# print("Raw pH value:", value)



In [18]:
# !pip install rasterio requests



In [19]:
import rasterio
from rasterio.windows import Window
from rasterio.transform import rowcol
import requests
import tempfile


lat = 30.563
lon = 31.083


def get_soil_value(layer):

    url = f"https://files.isric.org/soilgrids/latest/data/{layer}/"

    print("Layer:", layer)

    r = requests.get(url)

    print(r.status_code)


get_soil_value("phh2o")



Layer: phh2o


200


In [20]:
import requests

url = "https://files.isric.org/soilgrids/latest/data/phh2o/phh2o_0-5cm_mean/"

r = requests.get(url)

print(r.status_code)

print(r.text[:2000])



200
<!DOCTYPE html>
<html>
<head>
  <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
  <meta name="generator" content="WsgiDAV/4.3.3">
  <title>ISRIC - Index of /soilgrids/latest/data/phh2o/phh2o_0-5cm_mean/ </title>
  <link rel="shortcut icon" href="/:dir_browser/favicon.ico">
  <link rel="stylesheet" href="/:dir_browser/style.css" />
  <script defer src="/:dir_browser/script.js"></script>
  <style type="text/css"> A {behavior: url(#default#AnchorClick);} </style>
</head>

<body onload="onLoad()">

  <h1>
    <img class="logo" alt="ISRIC" title="ISRIC" src="/:dir_browser/logo.png">
    Webdav - Index of /soilgrids/latest/data/phh2o/phh2o_0-5cm_mean/
  </h1>
<p>
This page lists all resources made available through ISRIC - World Soil Information's Webdav functionality. 
This URL can also be accessed via the <a href="https://www.isric.org/explore/soilgrids/soilgrids-access">Webdav protocol</a>.
</p><p>
These resources are described in our data catalogue at <a href="htt

In [21]:
import requests
from bs4 import BeautifulSoup

url = "https://files.isric.org/soilgrids/latest/data/phh2o/phh2o_0-5cm_mean/"

html = requests.get(url).text

soup = BeautifulSoup(html, "html.parser")

for a in soup.find_all("a"):
    print(a.text.strip())



Webdav protocol
data.isric.org
www.isric.org
Mount
tileSG-000-019
tileSG-000-020
tileSG-000-021
tileSG-000-022
tileSG-000-023
tileSG-000-047
tileSG-000-048
tileSG-000-049
tileSG-000-050
tileSG-000-051
tileSG-000-056
tileSG-000-057
tileSG-001-017
tileSG-001-018
tileSG-001-019
tileSG-001-020
tileSG-001-021
tileSG-001-022
tileSG-001-023
tileSG-001-024
tileSG-001-045
tileSG-001-046
tileSG-001-047
tileSG-001-048
tileSG-001-050
tileSG-001-051
tileSG-001-054
tileSG-001-055
tileSG-001-056
tileSG-001-057
tileSG-001-058
tileSG-001-059
tileSG-001-060
tileSG-001-061
tileSG-001-062
tileSG-001-063
tileSG-002-010
tileSG-002-011
tileSG-002-012
tileSG-002-013
tileSG-002-014
tileSG-002-015
tileSG-002-016
tileSG-002-017
tileSG-002-018
tileSG-002-019
tileSG-002-020
tileSG-002-021
tileSG-002-022
tileSG-002-023
tileSG-002-024
tileSG-002-025
tileSG-002-044
tileSG-002-045
tileSG-002-046
tileSG-002-047
tileSG-002-049
tileSG-002-050
tileSG-002-051
tileSG-002-052
tileSG-002-053
tileSG-002-054
tileSG-002-055
tile

In [22]:
url = "https://files.isric.org/soilgrids/latest/data/phh2o/phh2o_0-5cm_mean/tileSG-016-071/"

html = requests.get(url).text

print(html[:2000])



<!DOCTYPE html>
<html>
<head>
  <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
  <meta name="generator" content="WsgiDAV/4.3.3">
  <title>ISRIC - Index of /soilgrids/latest/data/phh2o/phh2o_0-5cm_mean/tileSG-016-071/ </title>
  <link rel="shortcut icon" href="/:dir_browser/favicon.ico">
  <link rel="stylesheet" href="/:dir_browser/style.css" />
  <script defer src="/:dir_browser/script.js"></script>
  <style type="text/css"> A {behavior: url(#default#AnchorClick);} </style>
</head>

<body onload="onLoad()">

  <h1>
    <img class="logo" alt="ISRIC" title="ISRIC" src="/:dir_browser/logo.png">
    Webdav - Index of /soilgrids/latest/data/phh2o/phh2o_0-5cm_mean/tileSG-016-071/
  </h1>
<p>
This page lists all resources made available through ISRIC - World Soil Information's Webdav functionality. 
This URL can also be accessed via the <a href="https://www.isric.org/explore/soilgrids/soilgrids-access">Webdav protocol</a>.
</p><p>
These resources are described in our data

In [23]:
import rasterio

tif_url = "https://files.isric.org/soilgrids/latest/data/phh2o/phh2o_0-5cm_mean/tileSG-016-071/tileSG-016-071_1-1.tif"

with rasterio.open(tif_url) as src:

    print(src.crs)
    print(src.bounds)
    print(src.shape)



PROJCS["Interrupted_Goode_Homolosine",GEOGCS["GCS_unnamed ellipse",DATUM["unknown",SPHEROID["Unknown",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Interrupted_Goode_Homolosine"],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
BoundingBox(left=11912500.0, bottom=1250750.0, right=11976000.0, top=1400750.0)
(600, 254)


In [24]:
import rasterio

url = "https://files.isric.org/soilgrids/latest/data/phh2o/phh2o_0-5cm_mean/tileSG-016-071/tileSG-016-071_1-1.tif"

src = rasterio.open(url)

print(src.crs)
print(src.bounds)
print(src.shape)



PROJCS["Interrupted_Goode_Homolosine",GEOGCS["GCS_unnamed ellipse",DATUM["unknown",SPHEROID["Unknown",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Interrupted_Goode_Homolosine"],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
BoundingBox(left=11912500.0, bottom=1250750.0, right=11976000.0, top=1400750.0)
(600, 254)


In [25]:
# !pip install owslib rasterio



In [26]:
import os
import requests

url = "https://files.isric.org/soilgrids/latest/data_aggregated/1000m/phh2o/phh2o_0-5cm_mean_1000.tif"
filepath = "../data/raw/soilgrids/phh2o.tif"

if not os.path.exists(filepath):
    print("Downloading phh2o.tif...")
    r = requests.get(url)
    open(filepath,"wb").write(r.content)
    print("Downloaded")
else:
    print("phh2o.tif already exists, skipping download.")




phh2o.tif already exists, skipping download.


In [27]:
import rasterio
from rasterio.transform import rowcol
from pyproj import Transformer


lat = 30.563
lon = 31.083


with rasterio.open("../data/raw/soilgrids/phh2o.tif") as src:

    print("CRS:")
    print(src.crs)

    print("Bounds:")
    print(src.bounds)


    transformer = Transformer.from_crs(
        "EPSG:4326",
        src.crs,
        always_xy=True
    )


    x,y = transformer.transform(
        lon,
        lat
    )


    row,col = rowcol(
        src.transform,
        x,
        y
    )


    print("Pixel:")
    print(row,col)


    value = src.read(1)[row,col]


    print("Raw value:")
    print(value)


    print("Soil pH:")
    print(value/10)



CRS:


ESRI:54052
Bounds:
BoundingBox(left=-19949750.0, bottom=-6147999.999999999, right=19862250.0, top=8361000.000000001)
Pixel:
4958 23393


Raw value:
74
Soil pH:
7.4


In [28]:
import rasterio
from rasterio.transform import rowcol
from pyproj import Transformer
import numpy as np


def extract_soil_value(
    tif_path,
    lon,
    lat,
    scale=10
):

    with rasterio.open(tif_path) as src:

        transformer = Transformer.from_crs(
            "EPSG:4326",
            src.crs,
            always_xy=True
        )

        x, y = transformer.transform(
            lon,
            lat
        )


        row, col = rowcol(
            src.transform,
            x,
            y
        )


        value = src.read(1)[row, col]


        if value == src.nodata:
            return np.nan


        return float(value / scale)



In [29]:
ph = extract_soil_value(
    "../data/raw/soilgrids/phh2o.tif",
    31.083,
    30.563
)


print(ph)



7.4


In [30]:
import requests
import os

def download_soil_layer(layer, depth="0-5cm", resolution="1000m"):
    filename = f"../data/raw/soilgrids/{layer}.tif"
    if os.path.exists(filename):
        print(layer, "already exists ✔ (skipping download)")
        return filename
    url = f"https://files.isric.org/soilgrids/latest/data_aggregated/{resolution}/{layer}/{layer}_{depth}_mean_{resolution}.tif"
    print("Downloading", layer, "from", url)
    r = requests.get(url)
    if r.status_code != 200:
        print("Download failed:", url)
        return None
    with open(filename,"wb") as f:
        f.write(r.content)
    print(layer, "downloaded")
    return filename




In [31]:
def extract_soil_value(
    tif_file,
    lon,
    lat,
    scale=1
):

    with rasterio.open(tif_file) as src:


        transformer = Transformer.from_crs(
            "EPSG:4326",
            src.crs,
            always_xy=True
        )


        x,y = transformer.transform(
            lon,
            lat
        )


        row,col = rowcol(
            src.transform,
            x,
            y
        )


        value = src.read(1)[row,col]


        if value == src.nodata:
            return np.nan


        return float(value / scale)



In [32]:
layers = [
    "phh2o",
    "nitrogen",
    "soc",
    "clay",
    "sand",
    "silt"
]


files = {}


for layer in layers:

    files[layer] = download_soil_layer(
        layer
    )



phh2o already exists ✔ (skipping download)
nitrogen already exists ✔ (skipping download)
soc already exists ✔ (skipping download)
clay already exists ✔ (skipping download)
sand already exists ✔ (skipping download)
silt already exists ✔ (skipping download)


In [33]:
# lon = 31.083
# lat = 30.563


# soil_features = {}


# for layer,file in files.items():

#     value = extract_soil_value(
#         file,
#         lon,
#         lat,
#         scale=10 if layer=="phh2o" else 1
#     )

#     soil_features[layer] = value



# soil_features



In [34]:
import requests

url = "https://files.isric.org/soilgrids/latest/data_aggregated/1000m/nitrogen/"


r = requests.get(url)

print(r.status_code)

print(r.text[:2000])



200
<!DOCTYPE html>
<html>
<head>
  <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
  <meta name="generator" content="WsgiDAV/4.3.3">
  <title>ISRIC - Index of /soilgrids/latest/data_aggregated/1000m/nitrogen/ </title>
  <link rel="shortcut icon" href="/:dir_browser/favicon.ico">
  <link rel="stylesheet" href="/:dir_browser/style.css" />
  <script defer src="/:dir_browser/script.js"></script>
  <style type="text/css"> A {behavior: url(#default#AnchorClick);} </style>
</head>

<body onload="onLoad()">

  <h1>
    <img class="logo" alt="ISRIC" title="ISRIC" src="/:dir_browser/logo.png">
    Webdav - Index of /soilgrids/latest/data_aggregated/1000m/nitrogen/
  </h1>
<p>
This page lists all resources made available through ISRIC - World Soil Information's Webdav functionality. 
This URL can also be accessed via the <a href="https://www.isric.org/explore/soilgrids/soilgrids-access">Webdav protocol</a>.
</p><p>
These resources are described in our data catalogue at <a hre

In [35]:
import requests
from bs4 import BeautifulSoup


def get_tif_url(layer):

    base = f"https://files.isric.org/soilgrids/latest/data_aggregated/1000m/{layer}/"


    html = requests.get(base).text


    soup = BeautifulSoup(html,"html.parser")


    for a in soup.find_all("a"):

        href = a.get("href")

        if href and href.endswith(".tif"):
            return base + href


    return None



layers = [
    "phh2o",
    "nitrogen",
    "soc",
    "clay",
    "sand",
    "silt"
]


urls={}


for layer in layers:

    urls[layer]=get_tif_url(layer)

    print(layer, urls[layer])



phh2o https://files.isric.org/soilgrids/latest/data_aggregated/1000m/phh2o/phh2o_0-5cm_mean_1000.tif


nitrogen https://files.isric.org/soilgrids/latest/data_aggregated/1000m/nitrogen/nitrogen_0-5cm_mean_1000.tif


soc https://files.isric.org/soilgrids/latest/data_aggregated/1000m/soc/soc_0-5cm_mean_1000.tif


clay https://files.isric.org/soilgrids/latest/data_aggregated/1000m/clay/clay_0-5cm_mean_1000.tif


sand https://files.isric.org/soilgrids/latest/data_aggregated/1000m/sand/sand_0-5cm_mean_1000.tif


silt https://files.isric.org/soilgrids/latest/data_aggregated/1000m/silt/silt_0-5cm_mean_1000.tif


In [36]:
import os
import time
import requests
from tqdm import tqdm

files = {}

for layer, url in urls.items():

    if url is None:
        print(layer, "NOT FOUND")
        continue

    filename = f"../data/raw/soilgrids/{layer}.tif"

    # لو الملف موجود بالفعل
    if os.path.exists(filename):
        print(f"{filename} already exists ✔")
        files[layer] = filename
        continue

    success = False

    for attempt in range(5):

        try:

            response = requests.get(url, stream=True, timeout=120)
            response.raise_for_status()

            total = int(response.headers.get("content-length", 0))

            with open(filename, "wb") as f, tqdm(
                total=total,
                unit="B",
                unit_scale=True,
                desc=layer
            ) as bar:

                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))

            success = True
            files[layer] = filename
            print(f"{layer} Downloaded ✔")
            break

        except Exception as e:

            print(f"{layer}: Retry {attempt+1}/5 بسبب {e}")

            if os.path.exists(filename):
                os.remove(filename)

            time.sleep(5)

    if not success:
        print(f"{layer} Failed ❌")



../data/raw/soilgrids/phh2o.tif already exists ✔
../data/raw/soilgrids/nitrogen.tif already exists ✔
../data/raw/soilgrids/soc.tif already exists ✔
../data/raw/soilgrids/clay.tif already exists ✔
../data/raw/soilgrids/sand.tif already exists ✔
../data/raw/soilgrids/silt.tif already exists ✔


In [37]:
import rasterio
from pyproj import Transformer
from rasterio.transform import rowcol

def extract_soil_value(tif_path, lon, lat, scale=1):

    with rasterio.open(tif_path) as src:

        transformer = Transformer.from_crs(
            "EPSG:4326",
            src.crs,
            always_xy=True
        )

        x, y = transformer.transform(lon, lat)

        row, col = rowcol(src.transform, x, y)

        value = src.read(1)[row, col]

        if value == src.nodata:
            return None

        return float(value) / scale



In [38]:
import os

for file in [
    "../data/raw/soilgrids/phh2o.tif",
    "../data/raw/soilgrids/nitrogen.tif",
    "../data/raw/soilgrids/soc.tif",
    "../data/raw/soilgrids/clay.tif",
    "../data/raw/soilgrids/sand.tif",
    "../data/raw/soilgrids/silt.tif"
]:
    print(file, os.path.getsize(file)/1024/1024, "MB")



../data/raw/soilgrids/phh2o.tif 75.14319038391113 MB
../data/raw/soilgrids/nitrogen.tif 198.92382621765137 MB
../data/raw/soilgrids/soc.tif 214.18567276000977 MB
../data/raw/soilgrids/clay.tif 182.65748119354248 MB
../data/raw/soilgrids/sand.tif 189.61686325073242 MB
../data/raw/soilgrids/silt.tif 185.62142944335938 MB


In [39]:
import os
# os.remove("../data/raw/soilgrids/nitrogen.tif")
print("Bypassed nitrogen.tif removal to preserve local dataset")




Bypassed nitrogen.tif removal to preserve local dataset


In [40]:
import os
import requests
filepath = "../data/raw/soilgrids/nitrogen.tif"
if not os.path.exists(filepath):
    url = urls["nitrogen"]
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(filepath, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
    print("Finished")
else:
    print("nitrogen.tif already exists, skipping download.")




nitrogen.tif already exists, skipping download.


In [41]:
import os

print(os.path.getsize("../data/raw/soilgrids/nitrogen.tif")/1024/1024)



198.92382621765137


In [42]:
import os
import json
import time
import glob
import datetime
import gc
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import rowcol
from pyproj import Transformer
from concurrent.futures import ProcessPoolExecutor, as_completed

folders = ["../data/metadata", "../logs"]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

df_lucas = pd.read_csv("../data/features/lucas_labels.csv")
checkpoint_file = "../data/metadata/soilgrids_checkpoint.json"
chunk_dir = "../data/features/soilgrids_chunks"
os.makedirs(chunk_dir, exist_ok=True)
failed_points_file = "../data/metadata/failed_points.csv"
log_file = "../logs/soilgrids.log"

def log_msg(msg):
    timestamp = datetime.datetime.now().isoformat()
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] {msg}\n")
    print(msg)

start_idx = 0
if ENABLE_CHECKPOINT and os.path.exists(checkpoint_file):
    try:
        with open(checkpoint_file, "r") as f:
            cp = json.load(f)
            start_idx = cp.get("last_processed_idx", -1) + 1
        log_msg(f"Resuming SoilGrids from index {start_idx}")
    except Exception as e:
        log_msg(f"Failed to read checkpoint: {e}")

def process_point(row):
    point_id = int(row["POINT_ID"])
    lat = float(row["Latitude"])
    lon = float(row["Longitude"])
    date_str = str(row["Survey_Date"])
    meta = {
        "POINT_ID": point_id, "Latitude": lat, "Longitude": lon, "Survey_Date": date_str,
        "Extraction_Time": datetime.datetime.now().isoformat(), "Pipeline_Version": PIPELINE_VERSION,
        "SoilGrids_Version": "v2.0",
        "soil_ok": np.nan
    }
    try:
        soil_features = {}
        scales = {
            "phh2o": 10,
            "nitrogen": 100,
            "soc": 10,
            "clay": 10,
            "sand": 10,
            "silt": 10
        }
        for layer, fpath in files.items():
            with rasterio.open(fpath) as src:
                meta["CRS"] = str(src.crs)
                transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
                x, y = transformer.transform(lon, lat)
                r, c = rowcol(src.transform, x, y)
                val = src.read(1, window=rasterio.windows.Window(c, r, 1, 1))[0, 0]
                if val == src.nodata:
                    fallback_win = rasterio.windows.Window(c - 2, r - 2, 5, 5)
                    fallback_arr = src.read(1, window=fallback_win)
                    valid_vals = fallback_arr[fallback_arr != src.nodata]
                    if len(valid_vals) > 0:
                        val = valid_vals.mean()
                        soil_features[layer] = float(val) / scales[layer]
                    else:
                        soil_features[layer] = np.nan
                else:
                    soil_features[layer] = float(val) / scales[layer]
        meta.update(soil_features)
        meta["soil_ok"] = 1.0
        return {"status": "success", "data": meta}
    except Exception as e:
        err_msg = str(e)
        failed_row = {
            "POINT_ID": point_id, "longitude": lon, "latitude": lat, "date": date_str,
            "error_message": err_msg, "timestamp": datetime.datetime.now().isoformat()
        }
        pd.DataFrame([failed_row]).to_csv(
            failed_points_file, mode="a", header=not os.path.exists(failed_points_file), index=False
        )
        return {"status": "failed", "point_id": point_id, "error": err_msg}

def run_proc(row_dict):
    return process_point(row_dict)

num_samples = len(df_lucas)
start_time = time.time()
for c_idx in range(start_idx, num_samples, CHUNK_SIZE):
    chunk = df_lucas.iloc[c_idx : min(c_idx + CHUNK_SIZE, num_samples)]
    chunk_data = []
    log_msg(f"SoilGrids Processing batch {c_idx} to {min(c_idx+CHUNK_SIZE, num_samples)}...")
    import os
    if os.name == 'nt':
        from concurrent.futures import ThreadPoolExecutor
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(process_point, row): row for _, row in chunk.iterrows()}
            for fut in as_completed(futures):
                res = fut.result()
                if res["status"] == "success":
                    chunk_data.append(res["data"])
    else:
        try:
            rows_list = [row.to_dict() for _, row in chunk.iterrows()]
            with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
                futures = {executor.submit(run_proc, r): r for r in rows_list}
                for fut in as_completed(futures):
                    res = fut.result()
                    if res["status"] == "success":
                        chunk_data.append(res["data"])
        except Exception as e:
            log_msg(f"ProcessPool failed: {e}. Falling back to ThreadPoolExecutor...")
            from concurrent.futures import ThreadPoolExecutor
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                futures = {executor.submit(process_point, row): row for _, row in chunk.iterrows()}
                for fut in as_completed(futures):
                    res = fut.result()
                    if res["status"] == "success":
                        chunk_data.append(res["data"])


    if chunk_data:
        df_chunk = pd.DataFrame(chunk_data).drop_duplicates(subset=["POINT_ID"])
        df_chunk.to_csv(f"{chunk_dir}/soilgrids_part_{c_idx//CHUNK_SIZE + 1:03d}.csv", index=False)
    if ENABLE_CHECKPOINT:
        with open(checkpoint_file, "w") as f:
            json.dump({"last_processed_idx": min(c_idx + CHUNK_SIZE - 1, num_samples - 1)}, f)
    elapsed = time.time() - start_time
    processed = min(c_idx + CHUNK_SIZE, num_samples) - start_idx
    remaining = num_samples - min(c_idx + CHUNK_SIZE, num_samples)
    eta = (elapsed / processed) * remaining if processed > 0 else 0
    log_msg(f"Progress SoilGrids: {min(c_idx+CHUNK_SIZE, num_samples)}/{num_samples} ({processed/num_samples*100:.1f}%) | ETA: {eta/60:.1f} min")
    del chunk_data
    gc.collect()

all_chunks = sorted(glob.glob(f"{chunk_dir}/soilgrids_part_*.csv" ))
if all_chunks:
    df_final = pd.concat([pd.read_csv(ch) for ch in all_chunks], ignore_index=True)
    df_final.to_csv("../data/features/soilgrids_features.csv", index=False)
    log_msg(f"SoilGrids Merge Complete. Total rows: {len(df_final)}")






SoilGrids Processing batch 0 to 10...


Progress SoilGrids: 10/10 (100.0%) | ETA: 0.0 min
SoilGrids Merge Complete. Total rows: 10
